In [ ]:
from typing import TypedDict, Literal, Sequence
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thingking": {
            "type": "disabled"
        }
    }
)


#1. 状態を定義
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    ci_poem: str
    content_type: str


#2. ノードを定義
def node_a(state: OverAllState) -> OverAllState:
    poem = model.invoke([f"{state['topic']}をテーマにした詩を書いてください"]).content
    return {
        "poem": poem
    }


def node_b(state: OverAllState) -> OverAllState:
    joke = model.invoke([f"{state['topic']}をテーマにしたジョークを書いてください"]).content
    return {
        "joke": joke
    }


def node_c(state: OverAllState) -> OverAllState:
    ci_poem = model.invoke([f"{state['topic']}をテーマにした短歌を詠んでください"]).content
    return {
        "ci_poem": ci_poem
    }


def my_route(state: OverAllState) -> Sequence[Literal["poem", "joke", "ci_poem"]]:
    if "詩" in state["content_type"]:
        return ["poem", "ci_poem"]
    else:
        return ["joke", "ci_poem"]


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node(node_a)
builder.add_node(node_b)
builder.add_node(node_c)

builder.add_conditional_edges(
    START,
    my_route,
    path_map={
        "poem": "node_a",
        "joke": "node_b",
        "ci_poem": "node_c",
    }
)
builder.add_edge("node_b", END)
builder.add_edge("node_a", END)
builder.add_edge("node_c", END)

graph = builder.compile()
poem_res = graph.invoke({"topic": "猫", "content_type": "詩"})
print(poem_res)

joke_res = graph.invoke({"topic": "猫", "content_type": "ジョーク"})
print(joke_res)

from IPython.display import display

display(graph)


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from loguru import logger

from dotenv import load_dotenv

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thingking": {
            "type": "disabled"
        }
    }
)


class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str


def node_a(state: OverAllState) -> OverAllState:
    poem = model.invoke([HumanMessage(f"{state['topic']} をテーマにした俳句を詠んでください")]).content

    return {
        "poem": poem
    }


def node_b(state: OverAllState) -> OverAllState:
    joke = model.invoke([HumanMessage(f"{state['topic']} をテーマにしたジョークを書いてください")]).content

    return {
        "joke": joke
    }


def audit_node(state: OverAllState) -> OverAllState:
    logger.info(
        f"タスクノードがすべて実行完了しました。俳句は{'生成済み' if state['poem'] else '未生成'}、ジョークは{'生成済み' if state['joke'] else '未生成'}")


builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_node("audit_node", audit_node, defer=True)
builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge(START, "audit_node")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)
builder.add_edge("audit_node", END)

graph = builder.compile()
res = graph.invoke({"topic": "柴犬"})
print(res)

from IPython.display import display

display(graph)